# BertMS 质谱数据分析 - 重构版

本 notebook 使用模块化的代码结构进行质谱数据分析。

## 主要步骤

1. **数据加载和预处理** - 加载 MSP 文件并进行质量控制
2. **BERT 模型初始化和相似度计算** - 使用预训练的 BertMS 模型计算谱图相似度
3. **基础分子网络构建和聚类** - 构建网络并进行层次聚类分析
4. **增强注释** - 数据库搜索和结构预测
5. **约束聚类分析** - 应用大小和 m/z 约束的高级聚类

## 模块说明

所有核心功能已重构到 `bertms_analysis` 包中：
- `preprocessing`: 数据预处理
- `network`: 网络分析和聚类
- `annotation`: 数据库搜索和结构预测
- `utils`: 配置和日志工具

## 1. 导入依赖和配置

In [ ]:
# 标准库和第三方库
import os
import torch
import numpy as np
from tqdm import tqdm

# matchms 和 bertms 核心功能
from matchms_ws import load_from_msp, calculate_scores
from bertms_ws import Spec2BertVec
from model import BertForMaskedLM_pre

# 导入我们的分析模块
from bertms_analysis.utils import Config, setup_logger
from bertms_analysis.preprocessing import metadata_processing, peak_processing, is_valid_spectrum
from bertms_analysis.network import create_similarity_network, create_constrained_similarity_network
from bertms_analysis.annotation import (
    SpectrumDatabaseSearcher,
    StructuralClassPredictor,
    enhanced_cluster_annotation,
    visualize_enhanced_clusters
)

# 重新加载模块（确保使用最新代码）
import importlib
import bertms_analysis.annotation.database_search
importlib.reload(bertms_analysis.annotation.database_search)
from bertms_analysis.annotation.database_search import SpectrumDatabaseSearcher

# 初始化配置和日志
config = Config()
logger = setup_logger("bertms_analysis", level="INFO")
logger.info("BertMS 分析开始")

# 打印关键配置
print("\n=== 配置信息 ===")
print(f"设备: {config.device}")
print(f"批处理大小: {config.batch_size}")
print(f"相似度阈值: {config.similarity_cutoff}")
print(f"距离阈值: {config.distance_threshold}")
print(f"最大簇大小: {config.max_cluster_size}")
print(f"最大 m/z 差异: {config.max_mz_difference} Da")

## 2. 数据加载和预处理

加载 MSP 格式的质谱数据并进行预处理。

In [ ]:
# 数据文件路径（根据实际情况修改）
# 支持格式：.msp 或 .mgf
data_file = "RAW/252-5.msp"  # 或者 "your_file.mgf"

# 临时禁用详细日志，只显示关键信息
import logging
matchms_logger = logging.getLogger('matchms')
original_level = matchms_logger.level
matchms_logger.setLevel(logging.ERROR)  # 只显示错误

logger.info(f"从文件加载数据: {data_file}")

# 根据文件扩展名选择加载函数
if data_file.lower().endswith('.mgf'):
    from matchms_ws import load_from_mgf
    spectrums_raw = list(load_from_mgf(data_file))
    print(f"✓ 使用 MGF 格式加载")
elif data_file.lower().endswith('.msp'):
    from matchms_ws import load_from_msp
    spectrums_raw = list(load_from_msp(data_file))
    print(f"✓ 使用 MSP 格式加载")
else:
    raise ValueError(f"不支持的文件格式: {data_file}. 请使用 .msp 或 .mgf 文件")

print(f"✓ 原始数据: {len(spectrums_raw)} 条谱图")

# 元数据处理
logger.info("执行元数据处理...")
spectrums_metadata = [metadata_processing(s) for s in tqdm(spectrums_raw, desc="元数据处理")]

# 峰数据处理和质量控制
logger.info("执行峰数据处理...")
spectrums = []
for s in tqdm(spectrums_metadata, desc="峰数据处理"):
    processed = peak_processing(s)
    if processed is not None and is_valid_spectrum(processed):
        spectrums.append(processed)

# 恢复日志级别
matchms_logger.setLevel(original_level)

print(f"=== 数据加载完成 ===")
print(f"✓ 有效谱图: {len(spectrums)} 条")
print(f"✗ 过滤掉: {len(spectrums_raw) - len(spectrums)} 条无效谱图")
print(f"✓ 过滤率: {(1 - len(spectrums)/len(spectrums_raw))*100:.1f}%")

logger.info(f"数据预处理完成，有效谱图数量: {len(spectrums)}")

## 3. BERT 模型初始化

加载预训练的 BertMS 模型。

In [ ]:
# 配置和模型初始化
import os
import torch
from model import BertConfig, BertForMaskedLM_pre  # 重要：必须导入 BertConfig！
from bertms_ws import Spec2BertVec

# 更新已有的 config 对象，而不是重新创建
# config 已经在 Cell 1 中从 bertms_analysis.utils 导入并初始化了

# 更新项目路径和相关配置
config.project_dir = os.path.dirname(os.path.abspath('__file__'))  # 项目根目录（即本 notebook 所在目录）
config.dataset_dir = os.path.join(config.project_dir, 'data', 'ms')
config.pretrained_model_dir = os.path.join(config.project_dir, "bert_base_ms")  # 修复：直接拼接完整路径
config.data_name = 'ms'
config.device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
config.vocab_path = os.path.join(config.pretrained_model_dir, 'vocab.txt')
config.batch_size = 256
config.random_state = 82022
config.use_torch_multi_head = True
config.n_decimals = 2

# 添加必要的属性
config.use_embedding_weight = True
config.masked_rate = 0
config.masked_token_rate = 0
config.masked_token_unchanged_rate = 0
config.max_sen_len = None
config.pad_index = 0
config.is_sample_shuffle = True

# **重要：加载 BertConfig 并复制所有属性**
bert_config_path = os.path.join(config.pretrained_model_dir, "config.json")
try:
    bert_config = BertConfig.from_json_file(bert_config_path)
    # 复制所有 BertConfig 的属性到 config 对象
    for key, value in bert_config.__dict__.items():
        setattr(config, key, value)
    print(f"✓ 成功加载配置，vocab_size={config.vocab_size}")
except Exception as e:
    print(f"✗ 警告: 无法加载配置文件 {bert_config_path}: {e}")
    raise  # 如果加载失败就停止，不要使用错误的默认值

print(f"\n=== 配置检查 ===")
print(f"  vocab_size: {config.vocab_size}")
print(f"  hidden_size: {config.hidden_size}")
print(f"  distance_threshold: {config.distance_threshold}")
print(f"  similarity_cutoff: {config.similarity_cutoff}")

# 加载模型 - 使用 Windows 路径
model_path = os.path.join(
    config.pretrained_model_dir,
    "cache_20240325_215535",
    "model_ms_225.bin"
)

device = torch.device(config.device if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")

# 加载模型
model = BertForMaskedLM_pre.from_pretrained(config, model_path=model_path).to(device)
model.eval()
print("✓ 模型加载成功")

# 初始化 Spec2BertVec 转换器
bertms_similarity = Spec2BertVec(
    model=model,
    intensity_weighting_power=0.5,
    allowed_missing_percentage=5.0
)

print("✓ Spec2BertVec初始化成功")

## 4. 相似度计算

计算所有谱图对之间的相似度矩阵。

In [ ]:
logger.info("开始计算相似度矩阵...")

# 使用 BertMS 计算相似度
scores = calculate_scores(
    references=spectrums,
    queries=spectrums,
    similarity_function=bertms_similarity,
    is_symmetric=True
)

# 转换为 numpy 数组
scores_array = scores.to_array()

print(f"\n相似度矩阵形状: {scores_array.shape}")
print(f"相似度范围: [{scores_array.min():.4f}, {scores_array.max():.4f}]")
print(f"平均相似度: {scores_array.mean():.4f}")

logger.info(f"相似度计算完成，矩阵大小: {scores_array.shape}")

## 5. 基础分子网络构建和聚类

构建分子网络并进行层次聚类分析。

In [ ]:
logger.info("构建基础分子网络...")

# 创建分子网络
# 注意：create_similarity_network 返回一个元组，不是字典
network, cluster_labels, connected_components, precursor_mz_to_index = create_similarity_network(
    scores_array=scores_array,
    spectrums=spectrums,
    distance_threshold=config.distance_threshold,
    similarity_cutoff=config.similarity_cutoff,
    max_connections=config.max_connections,
    export_path="results/basic_network",
    mz_tolerance=config.mz_tolerance
)

# 获取 mz_to_merged 映射（需要重新调用 merge_duplicate_mz）
from bertms_analysis.network import merge_duplicate_mz
mz_to_merged, _, _ = merge_duplicate_mz(spectrums, config.mz_tolerance)

# 统计信息
n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise = list(cluster_labels).count(-1)

print(f"\n=== 基础网络统计 ===")
print(f"节点数: {network.number_of_nodes()}")
print(f"边数: {network.number_of_edges()}")
print(f"簇数量: {n_clusters}")
print(f"孤立节点: {n_noise}")
print(f"连通组件数: {len(connected_components)}")

logger.info(f"基础网络构建完成，{n_clusters} 个簇，{n_noise} 个孤立节点")